# ARC-AGI-3 P0 Random Agent Clean Baseline

Purpose: reproduce the simplest valid ARC-AGI-3 submission pattern outside `x_competition_files/`.

This is a reliability baseline, not an improvement attempt.

In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
%%writefile /kaggle/working/my_agent.py
import random
import time
from typing import Any, Iterable

from arcengine import FrameData, GameAction, GameState
from agents.agent import Agent


class MyAgent(Agent):
    """P0 random baseline agent.

    Goal: be simple, submission-safe, and easy to compare against later agents.
    This is not expected to be strong; it validates the submission path.
    """

    # Keep the official framework default for first clean baseline.
    # Future experiments can sweep this value deliberately.
    MAX_ACTIONS = 80

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        seed = int(time.time() * 1_000_000) + hash(self.game_id) % 1_000_000
        self.rng = random.Random(seed)

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        return latest_frame.state is GameState.WIN

    def _available_action_ids(self, latest_frame: FrameData) -> list[int]:
        raw_actions = getattr(latest_frame, "available_actions", None) or []
        ids: list[int] = []
        for action in raw_actions:
            try:
                action_id = action.value if hasattr(action, "value") else int(action)
            except Exception:
                continue
            if action_id != GameAction.RESET.value:
                ids.append(action_id)
        return sorted(set(ids))

    def _fallback_non_reset_actions(self) -> list[GameAction]:
        return [action for action in GameAction if action is not GameAction.RESET]

    def _random_non_reset_action(self, latest_frame: FrameData) -> GameAction:
        available_ids = self._available_action_ids(latest_frame)
        if available_ids:
            action_id = self.rng.choice(available_ids)
            try:
                return GameAction.from_id(action_id)
            except Exception:
                pass
        return self.rng.choice(self._fallback_non_reset_actions())

    def choose_action(self, frames: list[FrameData], latest_frame: FrameData) -> GameAction:
        if latest_frame.state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
            action = GameAction.RESET
            action.reasoning = "P0 baseline reset for NOT_PLAYED/GAME_OVER."
            return action

        action = self._random_non_reset_action(latest_frame)

        if action.is_simple():
            action.reasoning = f"P0 random baseline selected simple action {action.value}."
        elif action.is_complex():
            action.set_data({
                "x": self.rng.randint(0, 63),
                "y": self.rng.randint(0, 63),
            })
            action.reasoning = {
                "agent": "P0RandomAgent",
                "policy": "uniform_random_available_action_with_random_coordinate",
                "selected_action": str(action.value),
            }
        return action


In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for gateway to be ready.
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # Copy official agent framework to a writable location.
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    # Inject custom P0 agent.
    !cp /kaggle/working/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    # Minimal agent registry avoids eager imports of optional LLM/template dependencies.
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    \"myagent\": MyAgent,
}

__all__ = [\"Swarm\", \"Agent\", \"Playback\", \"AVAILABLE_AGENTS\", \"MyAgent\"]
""")

    # .env overrides .env.example defaults; main.py loads it second with override=True.
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent myagent

In [ ]:
# Non-rerun mode: produce a dummy submission so notebook commits cleanly on Kaggle.
import os
import pandas as pd

if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'],
    )
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    print('Non-rerun mode: wrote dummy submission.parquet')